<a href="https://colab.research.google.com/github/comparativechrono/computational_biology_notebooks/blob/main/pseudo_code_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Beverton–Holt simulation
Simulate the Beverton–Holt model with iterative and closed-form solutions.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Parameters for Beverton–Holt model
R = 3.0
x0 = 0.5
N = 20

# iterative solution
def simulate_beverton_holt(R, x0, N):
    xs = [x0]
    x = x0
    for t in range(N):
        x = (R * x) / (1 + x)
        xs.append(x)
    return np.array(xs)

# explicit solution
def beverton_holt_explicit(R, x0, N):
    ts = np.arange(0, N + 1)
    numerator = R - 1
    factor = (R - 1) / (x0 - 1)
    return numerator / (1 + factor * (R ** (-ts)))

iterative_sol = simulate_beverton_holt(R, x0, N)
explicit_sol  = beverton_holt_explicit(R, x0, N)

# compare both solutions
times = np.arange(0, N + 1)
plt.plot(times, iterative_sol, 'o-', label='Iterative')
plt.plot(times, explicit_sol, 'x--', label='Explicit')
plt.xlabel('Generation')
plt.ylabel('Population size')
plt.legend()
plt.title('Beverton–Holt model: iterative vs explicit solution')
plt.show()

# difference
print('Max absolute difference:', np.max(np.abs(iterative_sol - explicit_sol)))


### 2. Cobweb diagram
Generate the cobweb segments for a map and plot them.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Example map: use Beverton–Holt for demonstration
r = 2.9
f = lambda x: r * x / (1 + x)

# cobweb function
def cobweb_segments(f, x0, N):
    segments = []
    x_curr = x0
    for i in range(N):
        y_next = f(x_curr)
        segments.append(((x_curr, 0), (x_curr, y_next)))
        segments.append(((x_curr, y_next), (y_next, y_next)))
        x_curr = y_next
    return segments

x0 = 0.5
N_steps = 15
segments = cobweb_segments(f, x0, N_steps)

xs = np.linspace(0, 3, 400)
plt.plot(xs, f(xs), label='f(x)')
plt.plot(xs, xs, label='y=x')

for seg in segments:
    (x1, y1), (x2, y2) = seg
    plt.plot([x1, x2], [y1, y2], 'k-', lw=0.8)

plt.xlim(0, 3)
plt.ylim(0, 3)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Cobweb diagram')
plt.legend()
plt.show()


### 3. Equilibria and stability
Find fixed points of a map and classify stability.

In [ ]:

import numpy as np
from scipy.optimize import fsolve

f = lambda x: 3*x/(1+x)
f_prime = lambda x: 3/(1+x) - 3*x/(1+x)**2

guesses = [0.0, 1.0, 5.0]
roots = []
stability = []

for guess in guesses:
    root, info, ier, msg = fsolve(lambda x: f(x) - x, guess, full_output=True)
    if ier == 1:
        x_bar = root[0]
        if all(abs(x_bar - r) > 1e-6 for r in roots):
            roots.append(x_bar)
            st = 'stable' if abs(f_prime(x_bar)) < 1 else 'unstable'
            stability.append(st)

for x_bar, st in zip(roots, stability):
    print(f"Equilibrium at x̄ = {x_bar:.3f} is {st}. f'(x̄) = {f_prime(x_bar):.3f}")


### 4. Linearisation near equilibrium
Linearise a map around an equilibrium and simulate perturbations.

In [ ]:

import numpy as np

# Beverton–Holt function with R=3
def f(x):
    return 3 * x / (1 + x)

def f_prime(x):
    return 3/(1+x) - 3*x/(1+x)**2

x_bar = 2  # equilibrium R-1

deriv = f_prime(x_bar)
h0 = 0.1
N_steps = 10
h_values = []
x_values = []
h_current = h0

for t in range(N_steps + 1):
    h_values.append(h_current)
    x_values.append(x_bar + h_current)
    h_current = deriv * h_current

for t, (h_val, x_val) in enumerate(zip(h_values, x_values)):
    print(f"t={t}: h={h_val:.4f}, approx x={x_val:.4f}")


### 5. Two‑dimensional linear system
Solve J w_{t+1} = w_t using eigen decomposition.

In [ ]:

import numpy as np

J = np.array([[1.0, 1.0],
              [-1.0, 1.0]])
w0 = np.array([2.0, 4.0])

# eigenvalues and eigenvectors
w, v = np.linalg.eig(J)
v1 = v[:,0]
v2 = v[:,1]
coeffs = np.linalg.solve(np.column_stack((v1, v2)), w0)

def linear_solution(t):
    return coeffs[0] * (w[0]**t) * v1 + coeffs[1] * (w[1]**t) * v2

for t in range(6):
    print(f"t={t}: w_t =", linear_solution(t))

lambda_max = max(abs(w))
stability = 'stable' if lambda_max < 1 else ('unstable' if lambda_max > 1 else 'neutral')
print("Dominant eigenvalue magnitude:", lambda_max, "=>", stability)


### 6. Leslie matrix simulation
Construct a Leslie matrix and simulate population dynamics.

In [ ]:

import numpy as np

b = [1.0, 2.0]
s = [0.375]
m = len(b)

L = np.zeros((m, m))
L[0, :] = b
L[1:, :-1] = np.diag(s)

n0 = np.array([0.0, 4.0])
T = 10
populations = [n0]
for t in range(T):
    n_next = L @ populations[-1]
    populations.append(n_next)

for t, pop in enumerate(populations):
    print(f"t={t}: n = {pop}")


### 7. Dominant eigenvalue and stable age distribution
Compute the leading eigenvalue and normalised eigenvector of a Leslie matrix.

In [ ]:

import numpy as np

# reuse matrix L from previous cell
L = np.zeros((2,2))
L[0,:] = [1.0, 2.0]
L[1,0] = 0.375
w, v = np.linalg.eig(L)
idx = np.argmax(np.abs(w))
lambda1 = w[idx]
v1 = v[:, idx]
stable_age = v1 / np.sum(v1)
print("Dominant eigenvalue λ1:", lambda1)
print("Stable age distribution:", stable_age)


### 8. Discrete‑time Markov chain simulation
Simulate probabilities and compute equilibrium distribution.

In [ ]:

import numpy as np

Q = np.array([[0.5, 0.1], [0.5, 0.9]])
p0 = np.array([1.0, 0.0])
N = 10
p = p0.copy()
distributions = [p]
for t in range(1, N+1):
    p = Q @ p
    distributions.append(p)

for t, p in enumerate(distributions):
    print(f"t={t}: P(M)={p[0]:.3f}, P(H)={p[1]:.3f}")

w, v = np.linalg.eig(Q)
idx = np.argmin(np.abs(w - 1))
pi = v[:, idx]
pi = pi / np.sum(pi)
print("Equilibrium distribution π:", pi)


### 9. Linear mixed‑effects model
Fit a random intercept model using statsmodels.

In [ ]:

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2

np.random.seed(42)
num_groups = 10
n_per_group = 20

# random group intercepts
group_effects = np.random.normal(0, 2, size=num_groups)

beta0 = 1.0
beta1 = 0.5

rows = []
for g in range(num_groups):
    x = np.random.uniform(0, 10, size=n_per_group)
    y = beta0 + beta1 * x + group_effects[g] + np.random.normal(0, 1, size=n_per_group)
    group = np.array([g] * n_per_group)
    rows.append(pd.DataFrame({'y': y, 'x': x, 'group': group}))

df = pd.concat(rows, ignore_index=True)

# Fit models using statsmodels MixedLM
model_null  = smf.mixedlm("y ~ 1", df, groups=df["group"])
result_null = model_null.fit(method='lbfgs')

model_full  = smf.mixedlm("y ~ x", df, groups=df["group"])
result_full = model_full.fit(method='lbfgs')

print("Fixed effects (full model):
", result_full.params)
print("Random effect variance:", result_full.cov_re.iloc[0,0])

# Likelihood ratio test (ML fits)
model_null_ml  = smf.mixedlm("y ~ 1", df, groups=df["group"])
result_null_ml = model_null_ml.fit(method='lbfgs', reml=False)

model_full_ml  = smf.mixedlm("y ~ x", df, groups=df["group"])
result_full_ml = model_full_ml.fit(method='lbfgs', reml=False)

LLR = 2 * (result_full_ml.llf - result_null_ml.llf)
p_value = 1 - chi2.cdf(LLR, 1)
print(f"Likelihood ratio test statistic: {LLR:.3f}, p-value: {p_value:.3e}")


### 10. Random slopes mixed‑effects model
Fit a model with random intercepts and slopes.

In [ ]:

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

np.random.seed(1)
num_groups = 8
n_per_group = 30
beta0 = 2.0
beta1 = 1.0

random_intercepts = np.random.normal(0, 1.5, size=num_groups)
random_slopes    = np.random.normal(0, 0.5, size=num_groups)

rows = []
for g in range(num_groups):
    x = np.random.uniform(0, 5, size=n_per_group)
    y = beta0 + beta1 * x + random_intercepts[g] + random_slopes[g] * x + np.random.normal(0, 1, size=n_per_group)
    rows.append(pd.DataFrame({'y': y, 'x': x, 'group': g}))

df = pd.concat(rows, ignore_index=True)

# Fit random slopes model using MixedLM.from_formula with re_formula
md = sm.MixedLM.from_formula('y ~ x', groups='group', re_formula='1 + x', data=df)
result = md.fit(method='lbfgs')
print(result.summary())

# Extract correlation between intercept and slope
cov_re = result.cov_re
var_intercept = cov_re.iloc[0,0]
var_slope     = cov_re.iloc[1,1]
corr = cov_re.iloc[0,1] / np.sqrt(var_intercept * var_slope)
print(f"Correlation between random intercept and slope: {corr:.3f}")
